# BME780 Figure Dissection — Lab Notebook
**Megan McClean | Started: October 2026**

**Figure:** Figure 4 from Raychadhuri 2009 (*Biologist's Guide to Computational Genomics*, Ch. 4), applied to Chu et al. 1998 yeast sporulation data.

**What I'm trying to understand:** How PCA works — the linear algebra, the biological interpretation, and when it's the right tool vs. when you should use something else (t-SNE, UMAP).

---
*This notebook is my working document, not a polished write-up. It includes dead ends, questions I haven't fully answered, and things I figured out mid-stream.*

---
## `[2026-10-10]` First pass — trying to understand what PCA actually does

Watched the StatQuest PCA video (https://www.youtube.com/watch?v=_UVHneBUBW0). This was really helpful. Key things I took away:

- PCA rotates your data to a new coordinate system where the new axes (principal components) are ordered by how much variance they capture.
- PC1 is the direction that explains the most variance in the data, PC2 is orthogonal to PC1 and explains the next most, etc.
- The "loading" for each original variable (here: each timepoint) on a PC tells you how much that variable contributes to that PC.

What I still don't understand after the video:
- **How do you actually compute it?** The video talks about rotating the data but not the actual math.
- **What is an eigenvector and why does PCA use them?** The word came up but wasn't explained.
- **Why does it matter whether you center or scale the data?** The video says you should, but not what happens if you don't.

Going to work through the math next.

---
## `[2026-10-15]` Working through the linear algebra by hand

*(See photo: `sketches/pca_derivation_p1.jpg` and `sketches/pca_derivation_p2.jpg`)*

Pulled out my linear algebra notes. Key insight I finally got:

PCA is asking: "What is the direction $\mathbf{v}$ such that the projections of all my data points onto $\mathbf{v}$ have maximum variance?"

If your data matrix is **X** (n genes × p timepoints), after centering each column, the **covariance matrix** is:
$$\Sigma = \frac{1}{n-1} X^T X \quad \text{(p × p)}$$

Maximizing the projected variance subject to $|\mathbf{v}|=1$ gives you the **eigenvalue problem**: $\Sigma \mathbf{v} = \lambda \mathbf{v}$

So **PC directions = eigenvectors of the covariance matrix**. The eigenvalue $\lambda_k$ is the variance explained by PC $k$.

I worked through a tiny 3-gene × 2-timepoint example by hand to make sure I believed this. It worked.

**Still confused about:** What centering to do. I centered by gene (subtracted each gene's mean across timepoints) because that seemed right intuitively — each gene's "baseline" is different. But I'm not sure if that's what Raychadhuri does. Need to come back to this.

---
## `[2026-10-15]` First code attempt — getting the wrong answer

Tried to run PCA on the Chu et al. data structure. Got a weird result — my PC1 didn't match the "average expression" interpretation in Figure 4. Clustered genes that should be on the left were on the right.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.decomposition import PCA

np.random.seed(42)
timepoints = np.array([0, 0.5, 2, 5, 7, 11, 14], dtype=float)
T = len(timepoints)

def gaussian_pulse(t, peak, width, amplitude):
    return amplitude * np.exp(-0.5 * ((t - peak) / width) ** 2)

# Expression profiles
cluster_profiles = {
    'Early I':      gaussian_pulse(timepoints, 0.5, 1.2, 3.0),
    'Early II':     gaussian_pulse(timepoints, 1.0, 1.8, 2.5),
    'Early Middle': gaussian_pulse(timepoints, 2.5, 2.0, 2.0),
    'Middle':       gaussian_pulse(timepoints, 5.0, 2.0, 2.0),
    'Middle Late':  gaussian_pulse(timepoints, 7.5, 2.0, 1.8),
    'Late':         gaussian_pulse(timepoints, 12.0, 3.0, 2.2),
    'Metabolic':   -0.4 - 0.3 * timepoints / 14,
}

n_per_cluster = {'Early I': 130, 'Early II': 130, 'Early Middle': 90,
                 'Middle': 80, 'Middle Late': 50, 'Late': 40, 'Metabolic': 100}

all_expr, all_labels = [], []
for cluster, profile in cluster_profiles.items():
    n = n_per_cluster[cluster]
    scales = np.random.uniform(0.6, 1.4, (n, 1))
    all_expr.append(scales * profile + np.random.normal(0, 0.5, (n, T)))
    all_labels.extend([cluster] * n)

n_bg = 5500
all_expr.append(np.random.normal(0, 0.1, (n_bg, 1)) + np.random.normal(0, 0.2, (n_bg, T)))
all_labels.extend(['Background'] * n_bg)

X = np.vstack(all_expr)
labels = np.array(all_labels)

# WRONG APPROACH: row-centering (subtract each gene's mean across timepoints)
X_row_centered = X - X.mean(axis=1, keepdims=True)
pca_wrong = PCA(n_components=2)
scores_wrong = pca_wrong.fit_transform(X_row_centered)

print("WITH row-centering (wrong):")
for cl in ['Early I', 'Metabolic', 'Late']:
    mask = labels == cl
    print(f"  {cl}: PC1={scores_wrong[mask,0].mean():+.2f}, PC2={scores_wrong[mask,1].mean():+.2f}")

**What went wrong above:** Metabolic genes came out at positive PC1, same as Early I genes. That contradicts Figure 4 where Metabolic is on the left (negative PC1). 

**Why:** When I row-center (subtract each gene's mean across timepoints), I remove the information about average expression level. After row-centering, every gene has mean = 0 across timepoints. So PC1 can no longer separate "genes with high average expression" from "genes with low average expression" — that information was erased!

I had to read Ringnér (2008) to understand this. The key sentence: *"PCA is performed on the data matrix where genes are rows and conditions are columns, and the matrix is mean-centered over conditions."* 

"Mean-centered over conditions" means subtract the mean *of each condition (column/timepoint)* from all values in that column. This is what sklearn's PCA does by default (it centers by column). **Don't row-center if you want to preserve the average expression signal.**

---
## `[2026-10-20]` Fixed PCA + first real figure

In [ ]:
# CORRECT: sklearn PCA centers by COLUMN automatically — no manual step needed
pca = PCA(n_components=2)
scores = pca.fit_transform(X)

print(f"Variance explained: PC1={pca.explained_variance_ratio_[0]*100:.1f}%, "
      f"PC2={pca.explained_variance_ratio_[1]*100:.1f}%")
print(f"PC1 loadings: {np.round(pca.components_[0], 3)}")
print(f"PC2 loadings: {np.round(pca.components_[1], 3)}")
print()

print("WITH column-centering (correct):")
for cl in ['Early I', 'Metabolic', 'Late', 'Middle']:
    mask = labels == cl
    print(f"  {cl}: PC1={scores[mask,0].mean():+.2f}, PC2={scores[mask,1].mean():+.2f}")

**Now this makes sense!** 
- Early I: high PC1 (induced throughout → high average) ✓
- Metabolic: negative PC1 (repressed throughout) ✓ 
- Late: positive PC2 (expression increases over time) ✓

The PC1 loadings are `[0.58, 0.64, 0.48, 0.12, ...]` — all positive, roughly uniform. This confirms PC1 is approximately the average expression level.

The PC2 loadings go from negative (early timepoints) to positive (late timepoints): `[-0.17, -0.20, 0.18, 0.65, ...]`. Genes peaking late have positive projections → positive PC2 = "increasing trend".

In [ ]:
# Rough version of Figure 4 — just to see if the structure matches
cluster_colors = {
    'Early I': '#d62728', 'Early II': '#ff7f0e', 'Early Middle': '#bcbd22',
    'Middle': '#2ca02c', 'Middle Late': '#17becf', 'Late': '#1f77b4',
    'Metabolic': '#9467bd', 'Background': '#dddddd'
}

fig, ax = plt.subplots(figsize=(8, 6))
bg = labels == 'Background'
ax.scatter(scores[bg, 0], scores[bg, 1], c='#dddddd', s=3, alpha=0.3)
for cl in list(cluster_colors.keys())[:-1]:
    mask = labels == cl
    ax.scatter(scores[mask, 0], scores[mask, 1], c=cluster_colors[cl], s=15, alpha=0.8, label=cl)
    cx, cy = scores[mask, 0].mean(), scores[mask, 1].mean()
    ax.annotate(cl, (cx, cy), fontsize=7, ha='center',
                xytext=(0, 8), textcoords='offset points')

ax.set_xlabel('PC1 (Increasing Average Expression)')
ax.set_ylabel('PC2 (Increasing Positive Trend)')
ax.set_title('Quick figure — does it look like Raychadhuri Fig 4?')
ax.axhline(0, c='k', lw=0.5); ax.axvline(0, c='k', lw=0.5)
plt.tight_layout()
plt.savefig('figure4_draft_v1.png', dpi=120)
plt.show()
print("Saved. Cluster ordering looks right — Early I far right low, Late upper-left-ish, Metabolic lower-left.")

---
## `[2026-11-03]` SVD connection — finally clicked

*(See sketches/svd_pca_connection.jpg — I drew out the matrix factorizations to compare them side-by-side)*

SVD of X: $X = U \Sigma V^T$

PCA: principal component directions = columns of V (the right singular vectors). Scores = $XV = U\Sigma$.

They're the same rotation, described from different sides of the equation. PCA looks at it from the "what direction in feature space explains variance" angle; SVD gives you both the sample structure (U, the left singular vectors) and the feature structure (V) simultaneously.

Alter et al. (2000) used SVD directly on the Chu et al. data and called the columns of U "eigengenes" and the rows of V "eigenassays." The first eigengene was a smoothly rising function (average induction), the second was an early-then-late biphasic function (temporal trend). Same two signals as Raychadhuri's PC1 and PC2.

**Sanity check:** Let me verify that sklearn's PCA loadings match the right singular vectors from SVD.

In [ ]:
# Verify PCA == SVD (after mean-centering)
X_centered_col = X - X.mean(axis=0, keepdims=True)
U, s, Vt = np.linalg.svd(X_centered_col, full_matrices=False)

# PCA loadings should equal right singular vectors (up to sign)
pc1_pca = pca.components_[0]
pc1_svd = Vt[0]  # first row of Vt = first right singular vector

# Check alignment (may need sign flip — PCA sign is arbitrary)
sign = np.sign(np.dot(pc1_pca, pc1_svd))
print(f"PC1 from PCA:  {np.round(pc1_pca, 4)}")
print(f"PC1 from SVD:  {np.round(sign * pc1_svd, 4)}")
print(f"Max difference: {np.max(np.abs(pc1_pca - sign * pc1_svd)):.6f}")
print("\n✓ They match (within floating-point precision) — PCA and SVD give the same result")

---
## `[2026-11-08]` Polishing Figure 4 — adding the 95% ellipse

In [ ]:
import matplotlib.patches as mpatches
from matplotlib.patches import Ellipse
from scipy.stats import chi2

fig, axes = plt.subplots(1, 2, figsize=(14, 6))

for ax_idx, (ax, show_labels) in enumerate(zip(axes, [False, True])):
    bg = labels == 'Background'
    ax.scatter(scores[bg, 0], scores[bg, 1], c='#cccccc', s=3, alpha=0.3, zorder=1)
    for cl in list(cluster_colors.keys())[:-1]:
        mask = labels == cl
        ax.scatter(scores[mask, 0], scores[mask, 1],
                   c=cluster_colors[cl], s=10, alpha=0.7, zorder=2, label=cl)

    # 95% ellipse
    cov = np.cov(scores.T)
    vals, vecs = np.linalg.eigh(cov)
    order = vals.argsort()[::-1]
    vals, vecs = vals[order], vecs[:, order]
    chi2_val = chi2.ppf(0.95, df=2)
    width, height = 2 * np.sqrt(chi2_val * vals)
    angle = np.degrees(np.arctan2(*vecs[:, 0][::-1]))
    ellipse = Ellipse(xy=(scores[:, 0].mean(), scores[:, 1].mean()),
                      width=width, height=height, angle=angle,
                      edgecolor='black', facecolor='none', linewidth=1.5, zorder=5)
    ax.add_patch(ellipse)
    ax.text(0, scores[:, 1].mean() + height/2 + 0.1,
            '95% of genes', ha='center', fontsize=8)

    if show_labels:
        for cl in list(cluster_colors.keys())[:-1]:
            mask = labels == cl
            cx, cy = scores[mask, 0].mean(), scores[mask, 1].mean()
            ax.annotate(cl, (cx, cy), fontsize=8, fontweight='bold', ha='center',
                        xytext=(0, 6), textcoords='offset points')

    ax.set_xlabel('Component 1 (Increasing Average Expression)', fontsize=11)
    ax.set_ylabel('Component 2 (Increasing Positive Trend)', fontsize=11)
    ax.set_title(f'Panel {"A" if ax_idx==0 else "B"}')
    ax.axhline(0, c='k', lw=0.5); ax.axvline(0, c='k', lw=0.5)
    ax.set_xlim(-3, 6); ax.set_ylim(-3, 4)

    if ax_idx == 1:
        legend_handles = [mpatches.Patch(color=cluster_colors[c], label=c)
                          for c in list(cluster_colors.keys())[:-1]]
        ax.legend(handles=legend_handles, fontsize=8, loc='upper left')

fig.suptitle('Reproduction of Figure 4: Raychadhuri 2009 / Chu et al. 1998',
             fontsize=12, y=1.01)
plt.tight_layout()
plt.savefig('figure4_final.png', dpi=150, bbox_inches='tight')
plt.show()

---
## `[2026-11-14]` Thinking about whether I believe Raychadhuri's interpretation

Some things bother me:

**1. How much variance does PC1+PC2 capture?**
About 73% in my data. That means 27% of the variance is in PC3 and beyond — invisible in Figure 4. Genes that look close together in Figure 4 might actually be quite different in higher dimensions.

**2. The cluster separations in the figure are presented with drawn ellipses, not statistical confidence intervals.**
Looking at the actual scatter, there's substantial overlap between Early II and Early Middle, and between Middle and Middle Late. The figure LOOKS like clean separation partly because the ellipses are hand-drawn around the centroids. If I ran a classifier in this 2D space to predict cluster labels from PCA coordinates, I'd probably get a lot of misclassifications in those pairs.

**3. PC1 ≈ average expression is a property of this specific dataset, not PCA in general.**
If you used PCA on single-cell RNA-seq data, PC1 often captures library size (how many total counts a cell has), which is a technical artifact, not biology. The biological interpretability of PCA axes is not guaranteed — it's a property of the dominant variance structure in your data.

**Overall verdict:** The figure is a useful visualization and the qualitative story holds up. But I think it's presented with more certainty than is warranted. The real contribution of Chu et al. was the clustering, not the PCA — Raychadhuri is showing that the clusters are separable in this space, not that PCA would have found the clusters if you hadn't already known them.

---
## `[2026-11-16]` PCA limitations and when to use alternatives

Quick notes (not fully developed — writing this last-minute before submission):

**Why PCA became popular in genomics (1990s–2000s):**
- Microarray data: thousands of features (genes), few samples (conditions/timepoints)
- PCA gave the first real way to visualize genome-scale data in 2D
- Alter et al. 2000 showed that just 2–3 singular values explained most of the structure in this 6400-gene dataset
- Fast, interpretable, and the loadings are directly meaningful

**Where PCA breaks down:**
- Single-cell RNA-seq: 10,000+ cells, 20,000 genes, non-Gaussian distributions (lots of zeros), curved manifolds
- PCA is linear; real biological variation often lives on nonlinear manifolds
- The "crowding problem": projecting from high D → 2D with PCA compresses everything toward the origin

**Alternatives I want to understand better (didn't get to these yet):**
- t-SNE: nonlinear, preserves local structure, but inter-cluster distances meaningless
- UMAP: newer, faster, preserves more global structure than t-SNE, also nonlinear
- I want to run UMAP on this same Chu et al. data and see what happens

**Where PCA is still used:**
- QC for bulk RNA-seq: PCA is standard for detecting batch effects, outlier samples
- As a pre-processing step before t-SNE/UMAP (reduce to 50 PCs first, then run UMAP on those)
- Any dataset where linear structure dominates (timecourses, perturbation screens)

*(Note to self: find a single-cell dataset and try to show why PCA fails there vs. UMAP. Would make a good extension.)*

---
## `[2026-11-16]` Final reflection — what did I actually learn?

**Things I genuinely understand now that I didn't before:**
1. PCA = finding eigenvectors of the covariance matrix. I can do this by hand for a small matrix.
2. PCA and SVD give the same result — they're two descriptions of the same rotation.
3. The centering choice matters a lot. Row-centering vs. column-centering gives qualitatively different results. For this dataset, column-centering is correct.
4. PC1 capturing "average expression level" is a property of this specific dataset, not a universal property of PCA.

**Things I'm still confused about:**
- When exactly should you scale to unit variance (use the correlation matrix) vs. not scale (use the covariance matrix)? The intuition about unit mismatch makes sense, but the tradeoff is less clear to me when everything is already on the same scale.
- How to rigorously test whether the cluster separations in Figure 4 are statistically significant, not just visually apparent.

**Where I got stuck and wasted time:**
- The row-centering mistake took me a couple of hours. I kept re-running the code thinking there was a bug, when actually the problem was my conceptual understanding of what centering does.
- Setting up the chi-squared ellipse took longer than expected. I found three different formulas online and had to work out which one was correct.

**What I'd do next if I had more time:**
1. Run UMAP on the same Chu et al. data and compare
2. Try to cluster the data from PCA coordinates alone (without knowing the labels) and compare to Chu et al.'s clusters
3. Find a real single-cell dataset and show why PCA fails there

---

## References

1. Chu et al. 1998 (DOI: 10.1126/science.282.5389.699)
2. Raychadhuri et al. 2009 (DOI: 10.1142/9789814447331_0043)
3. Alter, Brown & Botstein 2000 (DOI: 10.1073/pnas.97.18.10101)
4. Ringnér, M. 2008 — What is principal component analysis? *Nature Biotechnology*, 26(3), 303–304.
5. StatQuest: PCA clearly explained (2015). https://www.youtube.com/watch?v=_UVHneBUBW0